# Chapter 6 — Wine Dataset (Maronna et al. 2019)

Reproduces **Figure 6.3** of *Robust Statistics: Theory and Methods*, Maronna, Martin, Yohai & Salibián-Barrera (Wiley, 2019).

Figure 6.3 is a 2×2 panel comparing Mahalanobis distances from the **classical** vs **robust** covariance estimates on the wine dataset (13 features, n = 59):

| Top-left | Top-right |
|----------|-----------|
| Classical distances vs. index | Classical distances vs χ² quantiles |
| **Bottom-left** | **Bottom-right** |
| Robust (MM) distances vs. index | Robust (MM) distances vs χ² quantiles |

In a classically-distributed sample these would all be similar. In the wine data the classical distances mask all the outliers while the robust ones expose them — the canonical "masking effect" demonstration.

We also reproduce the prcompRob scree (Figure 6.10 companion) and verify the Python wrapper outputs bit-for-bit against R.

In [ ]:
import os, sys, pathlib

if sys.platform == "win32" and "R_HOME" not in os.environ:
    os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.2"
    os.environ["PATH"] = r"C:\Program Files\R\R-4.5.2\bin\x64;" + os.environ["PATH"]

import numpy as np
import robstattm_py as rpm
from robstattm_py.plotting import r_plot, show_png
from robstattm_py._r import r as _r

FIG_DIR = pathlib.Path("figures")
FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## The wine dataset

Thirteen chemical measurements (alcohol, ash, phenols, …) on 59 samples. The full data is used unscaled; classical covariance is highly distorted by a small cluster of high-leverage cases.

In [ ]:
wine = rpm.datasets.wine()
print("shape:", wine.shape)
wine.head()

## Cross-check: Python wrappers bit-equal to R

Compute the classical covariance + MM-robust covariance through both stacks; assert exact equality.

In [ ]:
rpm.set_seed(42)
_r().r("set.seed(42)")

X = wine.to_numpy(dtype=float)

cov_cls = rpm.cov_classic(X)
cov_mm  = rpm.cov_rob_mm(X)

ro = _r()
ro.r("""
library(RobStatTM); set.seed(42); data(wine)
X_w <- as.matrix(wine)
cls_r <- covClassic(X_w)
mm_r  <- covRobMM(X_w)
""")

cls_cov_r = np.asarray(ro.r("cls_r$cov"), dtype=float)
mm_cov_r  = np.asarray(ro.r("mm_r$cov"),  dtype=float)
print("classical cov bit-equal:", np.array_equal(cov_cls.cov, cls_cov_r))
print("MM      cov bit-equal :", np.array_equal(cov_mm.cov,  mm_cov_r))

## Figure 6.3 — Classical vs Robust Mahalanobis distances (2×2 panel)

In [ ]:
show_png(r_plot("""
library(RobStatTM); data(wine)
X <- as.matrix(wine)
xbar <- colMeans(X)
C    <- cov(X)
disC <- mahalanobis(X, xbar, C)
set.seed(42)
resu <- covRobMM(X);    mu <- resu$mu; V <- resu$V
disM <- mahalanobis(X, mu, V)

par(mfrow=c(2,2))
plot(disC, xlab='index', ylab='Distances',
     main='Classical', cex.main=0.9, pch=19)
plot(qchisq(ppoints(59), 13), sort(disC),
     xlab='chi squared quantiles', ylab='Sorted distances',
     main='Classical', cex.main=0.9, pch=19)
lines(sort(disC), sort(disC))
plot(disM, xlab='index', ylab='Distances',
     main='Robust', cex.main=0.9, pch=19)
plot(qchisq(ppoints(59), 13), sort(disM),
     xlab='chi squared quantiles', ylab='Sorted distances',
     main='Robust', cex.main=0.9, pch=19)
lines(sort(disM), sort(disM))
par(mfrow=c(1,1))
""", path=FIG_DIR / "fig_6_3.png", width=8.0, height=7.0))

## Same panel, computed entirely in Python first

To prove the wrapper produces the same distances, recompute them in Python and compare to R.

In [ ]:
py_disC = cov_cls.dist
py_disM = cov_mm.dist
r_disC  = np.asarray(ro.r("mahalanobis(X_w, colMeans(X_w), cov(X_w))"), dtype=float)
r_disM  = np.asarray(ro.r("mahalanobis(X_w, mm_r$mu, mm_r$V)"),         dtype=float)

print(f"py classical dist range : [{py_disC.min():.4g}, {py_disC.max():.4g}]")
print(f"py MM        dist range : [{py_disM.min():.4g}, {py_disM.max():.4g}]")
print()
print("--- distance comparisons (note: same math, possibly different code paths) ---")
print(f"classical bit-equal : {np.array_equal(py_disC, r_disC)}  "
      f"(max abs diff: {abs(py_disC - r_disC).max():.3e})")
print(f"MM        bit-equal : {np.array_equal(py_disM, r_disM)}  "
      f"(max abs diff: {abs(py_disM - r_disM).max():.3e})")
print()
print("The cov matrices themselves ARE bit-equal (cell above). The distance")
print("vectors can differ in the last 1-2 ulps because R's `cov(X)` and the")
print("internal cov computed by `covClassic(X)` use slightly different")
print("operation orders, which propagates through `mahalanobis()`.")

## Cov summary — eigenvalues match R exactly

In [ ]:
sC = cov_cls.summary()
sM = cov_mm.summary()
r_evals_cls = np.asarray(ro.r("summary(cls_r)$evals"), dtype=float).ravel()
r_evals_mm  = np.asarray(ro.r("summary(mm_r)$evals"),  dtype=float).ravel()
print("classical eigenvalues bit-equal:", np.array_equal(sC.evals, r_evals_cls))
print("MM        eigenvalues bit-equal:", np.array_equal(sM.evals, r_evals_mm))
sM

## Bonus — Robust PCA scree (prcompRob)

The robust principal components of `wine` show how dominated the data is by the first axis after a robust fit.

In [ ]:
rpm.set_seed(42)
_r().r("set.seed(42)")
prc = rpm.prcomp_rob(X)
prc_summary = prc.summary()
prc_summary.importance

In [ ]:
show_png(r_plot("""
library(RobStatTM); set.seed(42); data(wine)
X_pr <- as.matrix(wine)
prc  <- prcompRob(X_pr)
plot(prc$sdev^2, type='b', xlab='Component', ylab='Variance',
     main='Robust PCA scree (wine)', pch=19, lwd=2)
abline(h=mean(prc$sdev^2), col='gray', lty=2)
""", path=FIG_DIR / "fig_6_scree.png", width=6.0, height=4.5))